# 实践项目 00：MNIST 手写数字分类

这份 Notebook 用手写数字图片完成一次完整的分类练习：先读取输入和标签，再检查数据，计算归一化参数，补全小型卷积神经网络，训练模型，评价测试集，最后观察输入加入噪声后结果怎样变化。

Kaggle Notebook 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按顺序运行单元格；下载到电脑运行是补充方式。每个任务都会说明输入、处理、输出和需要填写的位置。

图片和 JSON 用于帮助自己检查代码和记录结果。

**学生需要填写或修改的位置：** 带有 `TODO`、`None` 占位或“你的记录”的区域。只使用当前单元格提供的数据，不把参考结果数字直接写进代码。完成一个任务后，先运行当前单元格和后面的检查单元格。

## 任务总览

1. 读取图像和标签，核对 shape、数据类型、像素范围和类别分布。
2. 只用训练子集计算均值和标准差，并让三个数据集使用同一组参数。
3. 补全 CNN 的第二个卷积块和分类层。
4. 按“清空梯度 → 前向传播 → 损失 → 反向传播 → 更新参数”的顺序训练。
5. 用独立测试集计算准确率、每类 F1 和混淆矩阵。
6. 给测试图像加入高斯噪声，比较清洁输入和扰动输入。
7. 只改变学习率，完成一次单变量对照。

## 需要保存的结果

- `task0_data_visualization.png`
- `task0_training_curve.png`
- `task0_confusion_matrix.png`
- `task0_result.json`


## 实践顺序

每个任务都沿着“输入 → 处理 → 输出”的顺序进行。前一个任务产生的变量会成为后一个任务的输入，所以不要跳过中间的检查结果。

1. 读取 MNIST，得到图像张量和数字标签。
2. 检查图像 shape、dtype、像素范围和 0–9 标签数量。
3. 用训练子集统计量归一化，并划分训练集、验证集和测试集。
4. 让 CNN 把 `[1, 28, 28]` 图像变成 10 个类别分数。
5. 训练模型，用验证集选择模型，再用测试集做最终评价。
6. 加入噪声，观察输入受到干扰时准确率的变化。
7. 只改学习率，记录一次可解释的比较。

每个代码单元格运行后先读懂输出，再继续下一步。


In [ ]:
# 0. 导入库与固定随机性
# 输入：无；输出：后续步骤使用的库、随机种子、计算设备和工作目录。
from pathlib import Path
import json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print('device =', device)
print('working directory =', WORKDIR)


## 1. 读取数据

**输入：** Kaggle 中的 Digit Recognizer `train.csv`、`torchvision` 可以下载的 MNIST，或用于本地结构检查的小型回退数据。

**处理：** `load_digit_data()` 把每张图片整理为 `[1, 28, 28]` 的浮点张量，把标签整理为整数，并返回训练数据、测试数据和数据集名称。

**输出：** `train_full`、`test_dataset` 和 `dataset_name`。如果显示 `sklearn digits fallback`，只能说明 Notebook 结构可以检查；正式实践要在 Kaggle 中确认数据名称为 MNIST。


In [ ]:
def load_digit_data():
    """返回 train_dataset, test_dataset, dataset_name。图像范围统一为 [0, 1]，形状为 [1, 28, 28]。"""
    # 输入：可能存在的 CSV；输出：train_dataset、test_dataset、dataset_name。
    # Kaggle Digit Recognizer 常见路径
    csv_candidates = list(Path('/kaggle/input').glob('**/train.csv')) if Path('/kaggle/input').exists() else []
    for csv_path in csv_candidates:
        try:
            df = pd.read_csv(csv_path)
            pixel_cols = [c for c in df.columns if str(c).startswith('pixel')]
            if 'label' in df.columns and len(pixel_cols) == 784:
                x = torch.tensor(df[pixel_cols].to_numpy(), dtype=torch.float32).reshape(-1, 1, 28, 28) / 255.0
                y = torch.tensor(df['label'].to_numpy(), dtype=torch.long)
                g = torch.Generator().manual_seed(SEED)
                n_test = max(1000, int(0.15 * len(x)))
                perm = torch.randperm(len(x), generator=g)
                test_idx, train_idx = perm[:n_test], perm[n_test:]
                return TensorDataset(x[train_idx], y[train_idx]), TensorDataset(x[test_idx], y[test_idx]), 'MNIST / Kaggle Digit Recognizer'
        except Exception:
            pass

    # 标准 torchvision MNIST
    try:
        tfm = transforms.ToTensor()
        root = WORKDIR / 'mnist_data'
        train_ds = datasets.MNIST(root=root, train=True, transform=tfm, download=True)
        test_ds = datasets.MNIST(root=root, train=False, transform=tfm, download=True)
        return train_ds, test_ds, 'MNIST / torchvision'
    except Exception as exc:
        warnings.warn(f'MNIST 暂时无法读取，使用 sklearn digits 完成本地结构检查：{exc}')

    # 本地回退，仅用于检查 Notebook 结构
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    import torch.nn.functional as F
    d = load_digits()
    x = torch.tensor(d.images, dtype=torch.float32).unsqueeze(1) / 16.0
    x = F.interpolate(x, size=(28, 28), mode='bilinear', align_corners=False)
    y = torch.tensor(d.target, dtype=torch.long)
    idx = np.arange(len(y))
    tr, te = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=y.numpy())
    return TensorDataset(x[tr], y[tr]), TensorDataset(x[te], y[te]), 'sklearn digits fallback（仅环境检查）'

train_full, test_dataset, dataset_name = load_digit_data()
print(dataset_name)
print('train_full =', len(train_full), 'test =', len(test_dataset))


## 任务 1：数据检查与真实样本可视化

这一步先看清楚模型将要接收的输入。`train_full[0]` 给出一张图像和它的数字标签；`[1, 28, 28]` 中，`1` 是灰度通道，两个 `28` 分别是高度和宽度。

**学生需要填写：** 下一个代码单元格中的训练样本数、单张图像 shape、dtype、像素最小值、像素最大值和 0–9 类别计数。`class_counts` 应是长度为 10 的一维计数张量。

**输入与输出：** 输入是 `train_full`、`sample_image` 和 `all_labels`；输出是统计量、真实手写数字样本图和类别分布图。统计量必须来自当前数据。

**检查：** 运行后底部的 shape、像素范围和类别数量断言应通过，并应显示真实样本。


In [ ]:
# ===== 请在此处完成：项目00·任务1 数据检查统计量（开始） =====
# TODO 1：完成数据检查
sample_image, sample_label = train_full[0]

sample_count = None            # TODO：训练数据样本数
image_shape = None             # TODO：单张图像 shape
image_dtype = None             # TODO：数据类型
pixel_min = None               # TODO：像素最小值，转成 Python float
pixel_max = None               # TODO：像素最大值，转成 Python float

all_labels = torch.tensor([int(train_full[i][1]) for i in range(len(train_full))])
class_counts = None            # TODO：长度为 10 的类别计数张量
# ===== 请在此处完成：项目00·任务1 数据检查统计量（结束） =====

print('sample_count =', sample_count)
print('image_shape =', image_shape)
print('image_dtype =', image_dtype)
print('pixel range =', pixel_min, pixel_max)
print('label range =', int(all_labels.min()), int(all_labels.max()))
print('class_counts =', class_counts)

assert sample_count == len(train_full)
assert tuple(image_shape) == (1, 28, 28)
assert pixel_min >= 0 and pixel_max <= 1
assert len(class_counts) == 10

In [ ]:
# 输入：train_full 和上一单元格得到的 class_counts。
# 输出：真实样本图、标签分布图和 task0_data_visualization.png。
# 真实样本与类别分布
fig, axes = plt.subplots(3, 7, figsize=(12, 6))
for ax, idx in zip(axes.ravel()[:20], np.linspace(0, len(train_full)-1, 20, dtype=int)):
    image, label = train_full[idx]
    ax.imshow(image.squeeze().numpy(), cmap='gray')
    ax.set_title(f'label={int(label)}')
    ax.axis('off')
for ax in axes.ravel()[20:]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(WORKDIR/'task0_data_visualization.png', dpi=180, bbox_inches='tight')
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(np.arange(10), class_counts.numpy())
plt.xticks(np.arange(10))
plt.xlabel('digit label')
plt.ylabel('sample count')
plt.title('Training label distribution')
plt.show()


## 2. 划分训练集和验证集

训练集用于更新模型参数，验证集用于比较训练过程中的模型，测试集要留到所有设置确定后再使用。固定随机种子可以让这次划分保持一致。

**输入：** `train_full`。

**输出：** `train_dataset`、`val_dataset` 和保留的 `test_dataset`。下一个任务只能用 `train_dataset` 计算归一化参数。


In [ ]:
# 输入：train_full；输出：train_dataset、val_dataset 和保留的 test_dataset。
val_size = max(1000, int(0.15 * len(train_full))) if len(train_full) > 5000 else max(200, int(0.15 * len(train_full)))
train_size = len(train_full) - val_size
train_dataset, val_dataset = random_split(
    train_full, [train_size, val_size], generator=torch.Generator().manual_seed(SEED)
)
print('train =', len(train_dataset), 'validation =', len(val_dataset), 'test =', len(test_dataset))


## 任务 2：训练集归一化

归一化把像素变成以训练集为基准的数值，能让优化过程更稳定。验证集和测试集不能各自重新计算均值和标准差。

**学生需要填写：** 遍历 `stat_loader`，累计每批 `images` 的像素总和、平方和和像素数量，再计算 `train_mean` 与 `train_std`。标准差使用 `E[x²] - E[x]²`。

**输入与输出：** 每批输入 `images` 的 shape 是 `[batch, 1, 28, 28]`；输出 `train_mean`、`train_std` 是标量，`normalize_batch(images)` 返回同样 shape 的归一化张量。

**检查：** 均值和标准差只能来自训练子集；底部断言和后面的三个 DataLoader 创建应顺利运行。


In [ ]:
# ===== 请在此处完成：项目00·任务2 训练集均值与标准差（开始） =====
# TODO 2：从训练子集计算像素均值与标准差
stat_loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
pixel_sum = 0.0
pixel_sq_sum = 0.0
pixel_count = 0

for images, _ in stat_loader:
    # TODO：累计 images 的总和、平方和与像素数量
    pass

train_mean = None  # TODO
train_std = None   # TODO，使用 E[x²] - E[x]²
# ===== 请在此处完成：项目00·任务2 训练集均值与标准差（结束） =====
print('train_mean =', train_mean, 'train_std =', train_std)
assert 0 < train_mean < 1
assert train_std > 0

def normalize_batch(images):
    return (images - train_mean) / (train_std + 1e-8)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 任务 3：补全小型卷积神经网络

卷积层从局部图像区域提取特征，池化层缩小空间尺寸，全连接层把特征转换成 10 个类别分数。

**学生需要填写：** 在两个标记区域加入第二个卷积块和分类层，不改动 `forward` 或已有的第一块。

**输入与输出：** 模型输入是 `[batch, 1, 28, 28]`；第一块后是 `[batch, 16, 14, 14]`，第二块和池化后是 `[batch, 32, 7, 7]`，最终 logits 是 `[batch, 10]`。

**检查：** 查看模型结构、参数量和 `output shape`；最后的 shape 断言应通过。


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
# ===== 请在此处完成：项目00·任务3A 第二个卷积块（开始） =====
            # TODO 3A：加入 16→32 的 3×3 卷积、ReLU 和 2×2 最大池化
# ===== 请在此处完成：项目00·任务3A 第二个卷积块（结束） =====
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
# ===== 请在此处完成：项目00·任务3B 分类层（开始） =====
            # TODO 3B：加入 32×7×7 → 64 的全连接层、ReLU，以及 64 → 10 的输出层
# ===== 请在此处完成：项目00·任务3B 分类层（结束） =====
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SmallCNN().to(device)
example = torch.zeros(4, 1, 28, 28, device=device)
with torch.no_grad():
    output = model(example)
print(model)
print('output shape =', tuple(output.shape))
print('parameter count =', sum(p.numel() for p in model.parameters()))
assert tuple(output.shape) == (4, 10)

## 任务 4：补全训练步骤

一个训练批次依次完成：清空旧梯度、前向传播、计算损失、反向传播、更新参数。每轮训练结束后，验证集只用于评价，不更新参数。

**学生需要填写：** 在标记区域按上述顺序补全五个训练动作。

**输入与输出：** 输入是归一化后的 `images` `[batch, 1, 28, 28]` 和 `labels`；输出是标量 `loss`、批次预测和训练/验证指标。随后会保存训练曲线。

**检查：** 每轮都应打印 train loss、train accuracy 和 validation accuracy，并生成 `task0_training_curve.png`。


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(model, loader, noise_sigma=0.0):
    model.eval()
    ys, ps = [], []
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if noise_sigma > 0:
                images = torch.clamp(images + noise_sigma * torch.randn_like(images), 0, 1)
            logits = model(normalize_batch(images))
            total_loss += criterion(logits, labels).item() * len(labels)
            preds = logits.argmax(dim=1)
            ys.extend(labels.cpu().numpy())
            ps.extend(preds.cpu().numpy())
    return total_loss / len(loader.dataset), accuracy_score(ys, ps), np.array(ys), np.array(ps)

def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        images = normalize_batch(images)

# ===== 请在此处完成：项目00·任务4 标准训练步骤（开始） =====
        # TODO 4：完成五个训练动作
        # 1. optimizer 清空梯度
        # 2. model 前向传播得到 logits
        # 3. criterion 计算 loss
        # 4. loss 反向传播
        # 5. optimizer 更新参数
        pass
# ===== 请在此处完成：项目00·任务4 标准训练步骤（结束） =====

        running_loss += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return running_loss / total, correct / total

In [ ]:
# 输入：已补全的 model、train_loader、val_loader 和 evaluate。
# 输出：最佳验证模型、训练记录和 task0_training_curve.png。
EPOCHS = 20
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_state = None
best_val = -1

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f'Epoch {epoch+1}/{EPOCHS} | train loss {train_loss:.4f} | train acc {train_acc:.4f} | val acc {val_acc:.4f}')

model.load_state_dict(best_state)
model.to(device)

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(range(1, EPOCHS+1), history['train_loss'], marker='o', label='train loss')
ax1.plot(range(1, EPOCHS+1), history['val_loss'], marker='o', label='validation loss')
ax1.set_xlabel('epoch')
ax1.set_ylabel('loss')
ax2 = ax1.twinx()
ax2.plot(range(1, EPOCHS+1), history['val_acc'], marker='s', linestyle='--', label='validation accuracy')
ax2.set_ylabel('accuracy')
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='center right')
plt.title('Training history')
plt.tight_layout()
plt.savefig(WORKDIR/'task0_training_curve.png', dpi=180, bbox_inches='tight')
plt.show()


## 任务 5：测试集评价

测试集代表模型面对新输入时的表现。混淆矩阵的行是真实标签，列是预测标签；对角线上的数值表示预测正确的样本。

**学生需要填写：** 调用 `evaluate` 得到测试损失、准确率、真实标签和预测标签，再计算 macro-F1、每类 F1 和混淆矩阵。

**输入与输出：** 输入是最佳验证模型和 `test_loader`；输出是测试指标、长度为 10 的 `per_class_f1`、`[10, 10]` 的 `cm`，以及后面的混淆矩阵图片和分类报告。

**检查：** `cm.shape == (10, 10)` 和每类 F1 长度断言应通过。


In [ ]:
# ===== 请在此处完成：项目00·任务5 测试集指标（开始） =====
# TODO 5：调用 evaluate，并计算指标
# test_loss, test_acc, y_true, y_pred = ...
# macro_f1 = ...
# per_class_f1 = ...
# cm = ...

test_loss = None
test_acc = None
y_true = None
y_pred = None
macro_f1 = None
per_class_f1 = None
cm = None
# ===== 请在此处完成：项目00·任务5 测试集指标（结束） =====

print('test accuracy =', test_acc)
print('macro F1 =', macro_f1)
print('per-class F1 =', per_class_f1)

assert cm.shape == (10, 10)
assert len(per_class_f1) == 10

In [ ]:
# 输入：cm、y_true、y_pred；输出：混淆矩阵图片和分类报告。
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='viridis')
plt.colorbar(label='sample count')
plt.xticks(range(10))
plt.yticks(range(10))
plt.xlabel('predicted label')
plt.ylabel('true label')
plt.title('MNIST test confusion matrix')
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            plt.text(j, i, int(cm[i, j]), ha='center', va='center', fontsize=7,
                     color='white' if cm[i, j] > cm.max()*0.45 else 'black')
plt.tight_layout()
plt.savefig(WORKDIR/'task0_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()

report = classification_report(y_true, y_pred, digits=4)
print(report)


## 任务 6：噪声稳健性

这一步不再训练模型，只改变测试输入。代码给原始测试图像加入标准差为 `0.25` 的高斯噪声，并把像素裁剪回 `[0, 1]`，再用同一个模型预测。

**学生需要填写：** 使用 `evaluate(test_loader, noise_sigma=0.25)` 得到加噪准确率，并计算 `accuracy_drop = test_acc - noisy_acc`。

**输入与输出：** 输入是清洁测试图像和固定模型；输出是 `noisy_acc` 与 `accuracy_drop`。准确率下降越大，说明模型对这种输入扰动越敏感。

**检查：** 输出中应同时出现 clean accuracy、noisy accuracy 和 accuracy drop，`noisy_acc` 应在 `[0, 1]`。


In [ ]:
# ===== 请在此处完成：项目00·任务6 噪声稳健性评价（开始） =====
# TODO 6：使用 evaluate 的 noise_sigma 参数完成稳健性评价
noise_sigma = 0.25
noisy_loss = None
noisy_acc = None
accuracy_drop = None
# ===== 请在此处完成：项目00·任务6 噪声稳健性评价（结束） =====

print('clean accuracy =', test_acc)
print('noisy accuracy =', noisy_acc)
print('accuracy drop =', accuracy_drop)
assert noisy_acc <= 1 and noisy_acc >= 0

## 任务 7：单变量对照

对照实验一次只改一个条件。这里把基线学习率 `1e-3` 改成 `1e-2`，数据划分、模型结构、随机种子和训练轮数保持不变。

**学生需要填写：** 先在任务 4 的优化器代码中只修改学习率，重新运行训练单元格；再在本单元格记录改变的变量、基线值、新值、新设置的最佳验证准确率和观察。观察应说明比较条件、指标变化和可能原因。

**输入与输出：** 输入是前面训练产生的 `best_val` 和自己的新训练结果；输出是 `comparison` 字典。这份记录只用于理解单变量比较。


In [ ]:
# ===== 请在此处完成：项目00·任务7 单变量对照记录（开始） =====
# TODO 7：填写你的对照实验记录
comparison = {
    'changed_variable': None,
    'baseline_value': None,
    'new_value': None,
    'baseline_best_val_accuracy': float(best_val),
    'new_best_val_accuracy': None,
    'observation': None,
}
# ===== 请在此处完成：项目00·任务7 单变量对照记录（结束） =====
comparison

## 8. 保存结果

这一单元把数据名称、数据规模、模型设置、测试指标、噪声结果和单变量对照写入 `task0_result.json`。

**输入：** 前面单元格生成的变量，如 `dataset_name`、`best_val`、`test_acc`、`noisy_acc` 和 `comparison`。

**输出：** 一个可以重新查看的 JSON 文件。若显示 `sklearn digits fallback`，结果只用于结构检查；正式实践要在 Kaggle 中用 MNIST 重新运行。


In [ ]:
# 输入：前面各任务产生的指标和 comparison。
# 输出：task0_result.json。
result = {
    'dataset_name': dataset_name,
    'random_seed': SEED,
    'train_samples': len(train_dataset),
    'validation_samples': len(val_dataset),
    'test_samples': len(test_dataset),
    'image_shape': list(image_shape),
    'train_mean': float(train_mean),
    'train_std': float(train_std),
    'model_name': 'SmallCNN',
    'parameter_count': int(sum(p.numel() for p in model.parameters())),
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'optimizer': 'Adam',
    'learning_rate': 1e-3,
    'best_validation_accuracy': float(best_val),
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'test_macro_f1': float(macro_f1),
    'per_class_f1': [float(x) for x in per_class_f1],
    'noise_sigma': noise_sigma,
    'noisy_test_accuracy': float(noisy_acc),
    'accuracy_drop': float(accuracy_drop),
    'comparison': comparison,
    'output_files': [
        'task0_data_visualization.png',
        'task0_training_curve.png',
        'task0_confusion_matrix.png',
        'task0_result.json'
    ]
}
with open(WORKDIR/'task0_result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(json.dumps(result, ensure_ascii=False, indent=2))


## 结果检查

依次打开四个输出文件：数据图检查输入和标签，训练曲线检查参数是否发生学习，混淆矩阵定位容易混淆的数字，JSON 保存本次实验的设置和指标。

**学生需要记录：** 数据与标签检查、模型与训练曲线、测试表现与主要混淆、噪声影响和单变量对照结论。记录来自自己的实际输出。

<!-- ===== 请在此处完成：项目00·结果检查记录（开始） ===== -->
**你的记录：**

- 数据与标签检查：
- 模型与训练曲线：
- 测试表现与主要混淆：
- 噪声影响：
- 单变量对照结论：
<!-- ===== 请在此处完成：项目00·结果检查记录（结束） ===== -->
